### Shakespear Language Model - LSTM

1. Imports

In [ ]:
# These libraries handle text preprocessing, sequence padding, one-hot encoding, and model building.

import numpy as np
from tensorflow.keras.preprocessing.text import text_to_word_sequence, Tokenizer   # tokenization tools
from tensorflow.keras.preprocessing.sequence import pad_sequences                  # padding sequences
from tensorflow.keras.utils import to_categorical                                  # one-hot encoding
from tensorflow.keras.models import Sequential                                     # model container
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense               # neural network layers
from tensorflow.keras.optimizers import Adam                                       # optimizer

2. Load Text

In [ ]:
# LOAD AND READ THE TEXT

with open('data.txt', 'r', encoding='utf-8') as file:   # open the text file
    text = file.read()                                  # read entire file into a string

lines = text.lower().split('\n')                        # convert to lowercase and split into lines


3. Tokenization

In [ ]:
# TOKENIZATION

words = text_to_word_sequence(text)     # convert full text into a list of words

tokenizer = Tokenizer()                 # create tokenizer object
tokenizer.fit_on_texts(words)           # build vocabulary from all words

vocabulary_size = len(tokenizer.word_index) + 1   # total number of unique words + 1 for padding

sequences = tokenizer.texts_to_sequences(lines)   # convert each line into a list of integers


4. Building subsequence

In [ ]:
# BUILD SUBSEQUENCES

subsequences = []                                   # list to store all subsequences

for sequence in sequences:                          # loop through each line's sequence
    for i in range(1, len(sequence)):               # create growing subsequences
        subsequences.append(sequence[:i+1])         # append subsequence from start to i


5. Padding

In [ ]:
sequence_length = max(len(seq) for seq in sequences)   # find longest line length

sequences_padded = pad_sequences(
    subsequences,                                      # all subsequences
    maxlen=sequence_length,                            # pad to max length
    padding='pre'                                      # pad at the beginning
)


6. Preparing X and y 

In [ ]:
x = sequences_padded[:, :-1]                     # all tokens except last → input
y = sequences_padded[:, -1]                      # last token → label

y = to_categorical(y, num_classes=vocabulary_size)  # convert labels to one-hot vectors


7. Building Model

In [ ]:
model = Sequential()    # create empty model

# Embedding layer converts word IDs into dense vectors
model.add(Embedding(
    input_dim=vocabulary_size,          # vocabulary size
    output_dim=100,                     # embedding dimension
    input_length=sequence_length - 1    # input sequence length
))

# LSTM layer to learn long-term dependencies
model.add(LSTM(100))

# Dropout to reduce overfitting
model.add(Dropout(0.1))

# Output layer: softmax over all vocabulary words
model.add(Dense(vocabulary_size, activation='softmax'))

# Compile model with Adam optimizer and categorical crossentropy loss
model.compile(
    optimizer=Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()   # print model architecture


7. Training the model

In [ ]:
history = model.fit(
    x,                      # input sequences
    y,                      # one-hot encoded labels
    epochs=100,             # number of training epochs
    batch_size=128,         # batch size
    verbose=1               # show training progress
)


8. Returning the Model

In [ ]:
model